In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git transformers accelerate sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00


In [ ]:
import json
import requests
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
from pathlib import Path
from tqdm import tqdm
import torch

import clip
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from sentence_transformers import SentenceTransformer

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
ROOT_DIR = "/content/drive/MyDrive/CompNeuroscience-P1"

In [ ]:
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 500
OUTPUT_DIR = Path(ROOT_DIR) / "lamem_features2"
IMAGES_DIR = Path(ROOT_DIR) / "lamem_images"
DATA_JSON  = Path(ROOT_DIR) / "data.json"

# Original 5 emotions — unchanged
EMOTIONS = ["amusement", "excitement", "awe", "contentment", "sadness"]

OUTPUT_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)
print(f"Device: {DEVICE}")

### 4.1 CLIP (ViT-L/14)

In [ ]:
clip_model, clip_preprocess = clip.load("ViT-L/14", device=DEVICE)
clip_model.eval()
print("CLIP loaded")

### 4.2 Emotion prompts — softmax over fixed categories

Each emotion has multiple prompts. Similarities are averaged across prompts before softmax, reducing semantic noise and producing more robust continuous scores.

In [ ]:
# Multiple prompts per emotion → average similarities before softmax
EMOTION_PROMPTS = {
    "amusement":   [
        "a photo that conveys amusement",
        "a funny and amusing image",
        "a photo that makes you laugh or smile",
    ],
    "excitement":  [
        "a photo that conveys excitement",
        "an exciting and thrilling image",
        "a photo full of energy and excitement",
    ],
    "awe":         [
        "a photo that conveys awe",
        "an awe-inspiring and breathtaking image",
        "a photo that fills you with wonder",
    ],
    "contentment": [
        "a photo that conveys contentment",
        "a peaceful and serene image",
        "a photo that feels calm and satisfying",
    ],
    "sadness":     [
        "a photo that conveys sadness",
        "a melancholic and sorrowful image",
        "a photo that evokes feelings of sadness",
    ],
}

# Pre-compute text embeddings for all prompts, grouped by emotion
emotion_text_features = {}   # {emotion: Tensor (n_prompts, 768)}
with torch.no_grad():
    for emotion, prompts in EMOTION_PROMPTS.items():
        tokens = clip.tokenize(prompts).to(DEVICE)
        feats  = clip_model.encode_text(tokens)
        feats  = feats / feats.norm(dim=-1, keepdim=True)
        emotion_text_features[emotion] = feats

print("Emotion text features computed:",
      {k: v.shape for k, v in emotion_text_features.items()})

### 4.3 DINOv2 (ViT-L/14)

Complements CLIP: captures visual structure (texture, shape, composition) via self-supervised learning, without any textual supervision.

In [ ]:
dino_model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14")
dino_model = dino_model.to(DEVICE)
dino_model.eval()
print("DINOv2 loaded — output dim: 1024")

In [ ]:
from torchvision import transforms as T

dino_preprocess = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

### 4.4 BLIP-2

Used for three distinct free-form queries per image:
- **Caption**: visual description of the scene
- **Emotion description**: open-ended emotional response
- **Valence**: scalar derived from a natural language scale question
- **Arousal**: scalar derived from a natural language scale question

This avoids hardcoded priors — the model responds to what it actually sees.

In [ ]:
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
blip_model = torch.compile(blip_model)
blip_model.eval()
print("BLIP-2 loaded")

### 4.5 Sentence-BERT

In [ ]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim
print("SBERT loaded")

## 5 · Valence / Arousal parser

BLIP-2 answers scale questions in natural language. We map its response to a scalar in [-1, 1] by matching key phrases, with a fallback to 0.0 if the answer is ambiguous.

In [ ]:
# Ordered from most negative to most positive — first match wins
VALENCE_SCALE = [
    ("extremely negative", -1.00),
    ("very negative",      -0.75),
    ("negative",           -0.50),
    ("slightly negative",  -0.25),
    ("neutral",             0.00),
    ("slightly positive",   0.25),
    ("positive",            0.50),
    ("very positive",       0.75),
    ("extremely positive",  1.00),
]

AROUSAL_SCALE = [
    ("extremely calm",      -1.00),
    ("very calm",           -0.75),
    ("calm",                -0.50),
    ("slightly calm",       -0.25),
    ("neutral",              0.00),
    ("slightly exciting",    0.25),
    ("exciting",             0.50),
    ("very exciting",        0.75),
    ("extremely exciting",   1.00),
]

def parse_scale_response(text: str, scale: list[tuple[str, float]]) -> float:
    """Match the BLIP-2 free-text answer to the closest scale value."""
    text = text.lower().strip()
    for phrase, value in scale:
        if phrase in text:
            return value
    return 0.0   # fallback: neutral / ambiguous

## 6 · Helper functions

In [ ]:
def fetch_and_save_image(url: str, local_path: Path) -> Image.Image | None:
    if local_path.exists():
        return Image.open(local_path).convert("RGB")
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_bytes(r.content)
        return Image.open(BytesIO(r.content)).convert("RGB")
    except Exception:
        return None


def get_clip_features(image: Image.Image) -> np.ndarray:
    """Normalized CLIP image embedding (768,)."""
    tensor = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = clip_model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().float().numpy()[0]


def get_dino_features(image: Image.Image) -> np.ndarray:
    """Normalized DINOv2 image embedding (1024,)."""
    tensor = dino_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = dino_model(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().float().numpy()[0]


def get_emotion_scores(clip_feat: np.ndarray) -> dict:
    """
    Continuous emotion scores via prompt ensembling + softmax.
    Returns scores for each emotion and the dominant/secondary labels.
    """
    feat = torch.tensor(
        clip_feat, dtype=emotion_text_features["amusement"].dtype
    ).unsqueeze(0).to(DEVICE)   # (1, 768)

    logits = []
    for emotion in EMOTIONS:
        proto = emotion_text_features[emotion]        # (n_prompts, 768)
        sim   = (feat @ proto.T).mean(dim=-1)         # scalar per emotion
        logits.append(sim)

    logits = torch.cat(logits)                        # (n_emotions,)
    scores = logits.softmax(dim=-1).cpu().float().numpy()

    emotion_scores = dict(zip(EMOTIONS, scores.tolist()))
    sorted_idx     = scores.argsort()[::-1]
    dominant       = EMOTIONS[sorted_idx[0]]
    secondary      = EMOTIONS[sorted_idx[1]]

    return {
        "emotion_scores": emotion_scores,
        "dominant_emotion": dominant,
        "secondary_emotion": secondary,
    }


def blip_query(image: Image.Image, question: str, max_new_tokens: int = 60) -> str:
    """
    Run a single BLIP-2 VQA query on an image and return the decoded answer.
    Uses Question Answering mode (question is passed as text prompt).
    """
    inputs = blip_processor(
        images=image,
        text=question,
        return_tensors="pt"
    ).to(DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32)
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return blip_processor.decode(out[0], skip_special_tokens=True).strip()


def get_blip_features(image: Image.Image) -> dict:
    """
    Run all three BLIP-2 queries for a single image:
      - caption:             visual description of the scene
      - emotion_description: open-ended emotional response (free vocabulary)
      - valence_raw / valence: sentiment scale question → scalar in [-1, 1]
      - arousal_raw / arousal: energy scale question    → scalar in [-1, 1]
    """
    caption = blip_query(
        image,
        "Describe what is happening in this image.",
        max_new_tokens=60,
    )

    emotion_description = blip_query(
        image,
        "How does this image make you feel emotionally? Describe the feelings it evokes.",
        max_new_tokens=60,
    )

    valence_raw = blip_query(
        image,
        ("On a scale from extremely negative to extremely positive, "
"how does this image feel? "
"Choose one: extremely negative, very negative, negative, slightly negative, "
"neutral, slightly positive, positive, very positive, extremely positive."),
        max_new_tokens=15,
    )

    arousal_raw = blip_query(
        image,
        ("On a scale from extremely calm to extremely exciting, "
"how energetic or stimulating is this image? "
"Choose one: extremely calm, very calm, calm, slightly calm, "
"neutral, slightly exciting, exciting, very exciting, extremely exciting."),
        max_new_tokens=15,
    )

    return {
        "caption":             caption,
        "emotion_description": emotion_description,
        "valence_raw":         valence_raw,
        "valence":             parse_scale_response(valence_raw, VALENCE_SCALE),
        "arousal_raw":         arousal_raw,
        "arousal":             parse_scale_response(arousal_raw, AROUSAL_SCALE),
    }


def save_batch(
    records:    list[dict],
    clip_vecs:  list[np.ndarray],
    dino_vecs:  list[np.ndarray],
    sbert_vecs: list[np.ndarray],
    batch_idx:  int,
):
    df   = pd.DataFrame(records)
    path = OUTPUT_DIR / f"batch_{batch_idx:04d}.parquet"
    df.to_parquet(path, index=False)

    np.save(OUTPUT_DIR / f"batch_{batch_idx:04d}_clip.npy",  np.stack(clip_vecs))
    np.save(OUTPUT_DIR / f"batch_{batch_idx:04d}_dino.npy",  np.stack(dino_vecs))
    np.save(OUTPUT_DIR / f"batch_{batch_idx:04d}_sbert.npy", np.stack(sbert_vecs))

    print(f"  Saved {len(records)} records → {path.name}")

## 7 · SBERT encoding strategy

Each image produces two free-text fields from BLIP-2:
- `caption`: visual scene description
- `emotion_description`: open-ended emotional response

We concatenate them before encoding so the SBERT vector captures both semantic content and emotional tone in a single 384-d embedding. This is saved as `sbert_embeddings.npy`.

## 8 · Main pipeline

In [ ]:
with open(DATA_JSON) as f:
    data = json.load(f)
print(f"Total images in JSON: {len(data)}")

In [ ]:
done_files  = sorted(OUTPUT_DIR.glob("batch_*.parquet"))
done_count  = sum(len(pd.read_parquet(p)) for p in done_files)
batch_num   = len(done_files)
print(f"Already processed: {done_count} images. Resuming from index {done_count}.")

In [ ]:
from PIL import Image

meta_buffer  = []
clip_buffer  = []
dino_buffer  = []
sbert_buffer = []

for item in tqdm(data[done_count:], initial=done_count, total=len(data)):
    local_path = IMAGES_DIR / item["name"]

    try:
        if not local_path.exists():
            raise FileNotFoundError(f"{local_path} not found")

        image = Image.open(local_path).convert("RGB")

    except Exception as e:
        image = None

    if image is None:
        meta_buffer.append({
            "name":             item["name"],
            "local_path":       str(local_path),
            "dataset":          item.get("dataset", ""),
            "memscore":         item.get("memscore", None),
            "original_emotion": item.get("emotion", ""),
            "error":            True,
        })
        clip_buffer.append(np.zeros(768,  dtype=np.float32))
        dino_buffer.append(np.zeros(1024, dtype=np.float32))
        sbert_buffer.append(np.zeros(384, dtype=np.float32))

    else:
        clip_feat   = get_clip_features(image)
        dino_feat   = get_dino_features(image)
        emo         = get_emotion_scores(clip_feat)
        blip        = get_blip_features(image)

        combined_text = blip["caption"] + " | " + blip["emotion_description"]
        sbert_feat    = sbert_model.encode(combined_text).astype(np.float32)

        meta_buffer.append({
            "name":                 item["name"],
            "local_path":           str(local_path),
            "dataset":              item.get("dataset", ""),
            "memscore":             item.get("memscore", None),
            "original_emotion":     item.get("emotion", ""),
            # Continuous emotion scores (CLIP softmax)
            **{f"emotion_{e}": emo["emotion_scores"][e] for e in EMOTIONS},
            "dominant_emotion":     emo["dominant_emotion"],
            "secondary_emotion":    emo["secondary_emotion"],
            # Free-form BLIP-2 outputs
            "caption":              blip["caption"],
            "emotion_description":  blip["emotion_description"],
            # Valence and Arousal
            "valence_raw":          blip["valence_raw"],
            "valence":              blip["valence"],
            "arousal_raw":          blip["arousal_raw"],
            "arousal":              blip["arousal"],
            "error":                False,
        })
        clip_buffer.append(clip_feat)
        dino_buffer.append(dino_feat)
        sbert_buffer.append(sbert_feat)

    if len(meta_buffer) >= BATCH_SIZE:
        save_batch(meta_buffer, clip_buffer, dino_buffer, sbert_buffer, batch_num)
        batch_num   += 1
        meta_buffer  = []
        clip_buffer  = []
        dino_buffer  = []
        sbert_buffer = []

if meta_buffer:
    save_batch(meta_buffer, clip_buffer, dino_buffer, sbert_buffer, batch_num)

print("Pipeline complete.")

## 9 · Consolidate final artifacts

In [ ]:
print("Concatenating metadata...")
all_dfs  = [pd.read_parquet(p) for p in sorted(OUTPUT_DIR.glob("batch_*.parquet"))]
final_df = pd.concat(all_dfs, ignore_index=True)
final_df.to_parquet(OUTPUT_DIR / "lamem_features_full.parquet", index=False)
print(f"Parquet saved — shape: {final_df.shape}")

In [ ]:
print("Concatenating CLIP embeddings...")
clip_matrix = np.concatenate(
    [np.load(p) for p in sorted(OUTPUT_DIR.glob("batch_*_clip.npy"))], axis=0
)
np.save(OUTPUT_DIR / "clip_embeddings.npy", clip_matrix)
print(f"CLIP embeddings — shape: {clip_matrix.shape}")

In [ ]:
print("Concatenating DINO embeddings...")
dino_matrix = np.concatenate(
    [np.load(p) for p in sorted(OUTPUT_DIR.glob("batch_*_dino.npy"))], axis=0
)
np.save(OUTPUT_DIR / "dino_embeddings.npy", dino_matrix)
print(f"DINO embeddings — shape: {dino_matrix.shape}")

In [ ]:
print("Concatenating SBERT embeddings...")
sbert_matrix = np.concatenate(
    [np.load(p) for p in sorted(OUTPUT_DIR.glob("batch_*_sbert.npy"))], axis=0
)
np.save(OUTPUT_DIR / "sbert_embeddings.npy", sbert_matrix)
print(f"SBERT embeddings — shape: {sbert_matrix.shape}")

## 10 · Summary

In [ ]:
print("=" * 60)
print("SAVED ARTIFACTS")
print("=" * 60)
print(f"lamem_features_full.parquet   shape: {final_df.shape}")
print(f"clip_embeddings.npy           shape: {clip_matrix.shape}")
print(f"dino_embeddings.npy           shape: {dino_matrix.shape}")
print(f"sbert_embeddings.npy          shape: {sbert_matrix.shape}")
print("-" * 60)
print(f"Images saved locally:         {len(list(IMAGES_DIR.rglob('*.jpg')))}")
print(f"Fetch errors:                 {final_df['error'].sum()}")
print("-" * 60)
print("Parquet columns:")
for col in final_df.columns:
    print(f"  {col}")
print("=" * 60)
print("\nSample — emotions + valence/arousal:")
emo_cols = [f"emotion_{e}" for e in EMOTIONS]
display_cols = ["name", "dominant_emotion", "secondary_emotion",
                "valence", "arousal", "valence_raw", "arousal_raw",
                "emotion_description"] + emo_cols
print(final_df[display_cols].head(3).to_string())

## 11 · Release runtime

In [ ]:
from google.colab import runtime
runtime.unassign()